# Chapter 13. 파이썬 표준 라이브러리 심화

이 노트북은 [Chapter 13 원본 문서](../doc/Chapter%2013.%20%ED%8C%8C%EC%9D%B4%EC%8D%AC%20%ED%91%9C%EC%A7%80%EB%93%B1%20%EB%9D%BC%EC%9D%B4%EB%B8%8C%EB%9F%AC%EB%A6%AC%20%EC%8B%AC%ED%99%94.md)를 실습용으로 변환한 자료입니다.

`reprlib`, `pprint`, `textwrap`, `Template`, `struct`, `threading`, `logging`, `weakref`, `array`, `deque`, `bisect`, `heapq`, `Decimal`까지 함께 실습합니다.

## 1. 출력 형식 개선

대형 데이터 구조를 그대로 출력하면 복잡해지므로, 화면에 적합한 형태로 보기 좋게 정리하는 도구를 사용합니다.

In [ ]:
import reprlib
from pprint import pprint
import textwrap

large_prices = {f"COIN{index}": index * 100 for index in range(20)}
print(reprlib.repr(large_prices))

portfolio = {
    "BTC": {"quantity": 0.012, "average_price": 105_000_000},
    "ETH": {"quantity": 0.35, "average_price": 3_500_000},
}
pprint(portfolio, width=50)

message = "가격 데이터 수집이 완료되었습니다. 최근 24시간 동안 거래량과 변동률을 계산하고 매매 신호를 확인했습니다."
print(textwrap.fill(message, width=35))

## 2. 사용자 수정 가능한 템플릿

`string.Template`은 사용자 입력을 반영하는 문구를 안전하게 관리할 때 유용합니다.

In [ ]:
from string import Template

report_template = Template("$symbol 가격은 $price 원이며, 변동률은 $change_rate%입니다.")
report = report_template.substitute(symbol="BTC", price="105,000,000", change_rate="2.35")
print(report)

warning_template = Template("$symbol 거래 수수료는 $$${fee}입니다.")
print(warning_template.substitute(symbol="BTC", fee="4,200"))

missing_template = Template("$symbol의 가격은 $price원입니다.")
print(missing_template.safe_substitute(symbol="ETH"))

## 3. 바이너리 레코드 처리: `struct`

고정 길이 바이너리 데이터를 읽고 쓰는 표준 방식입니다.

In [ ]:
import struct

packed = struct.pack("<IQH", 1, 105_000_000, 12)
print(packed)
print(len(packed))
print(struct.unpack("<IQH", packed))

record = struct.pack("<8sQ", b"BTC", 105_000_000)
code, price = struct.unpack("<8sQ", record)
print(code.rstrip(b"\x00").decode("ascii"), price)

## 4. 멀티스레딩과 `queue`

동시에 실행할 수 있는 작업은 스레드로 분리하고, 작업을 안전하게 전달하기 위해 큐를 사용합니다.

In [ ]:
import threading
from queue import Queue


def collect_price(symbol):
    print(f"{symbol} 가격 수집 시작")
    print(f"{symbol} 가격 수집 완료")


threads = [threading.Thread(target=collect_price, args=(symbol,)) for symbol in ["BTC", "ETH", "XRP"]]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()


def worker(tasks):
    while True:
        symbol = tasks.get()
        if symbol is None:
            tasks.task_done()
            break
        print(f"{symbol} 작업 처리")
        tasks.task_done()


queue = Queue()
worker_thread = threading.Thread(target=worker, args=(queue,))
worker_thread.start()
for symbol in ["BTC", "ETH", "XRP"]:
    queue.put(symbol)
queue.put(None)
queue.join()
worker_thread.join()
print("작업 큐 종료")

## 5. 로깅: `logging`

프로덕션 프로그램에서는 `print`보다 `logging`이 더 안정적이고 추적하기 쉽습니다.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
logger = logging.getLogger("trading")
logger.info("가격 데이터 수집 완료")
logger.warning("거래량 데이터가 부족합니다.")
logger.error("주문 처리에 실패했습니다.")

price = 105_000_000
logger.info("현재 가격: %s", price)

## 6. 약한 참조와 리스트 보조 자료구조

캐시와 우선순위 처리에 적합한 도구들을 살펴봅니다.

In [ ]:
import gc
import weakref
from array import array
from collections import deque
import bisect
from heapq import heapify, heappop, heappush


class PriceSnapshot:
    def __init__(self, symbol, price):
        self.symbol = symbol
        self.price = price

    def __repr__(self):
        return f"{self.symbol}({self.price})"


snapshot = PriceSnapshot("BTC", 105_000_000)
cache = weakref.WeakValueDictionary()
cache["latest"] = snapshot
print(cache["latest"])
del snapshot
gc.collect()
print("latest" in cache)

volumes = array("I", [4000, 10, 700, 22222])
print(sum(volumes), volumes[1:3])

orders = deque(["order-1", "order-2", "order-3"])
orders.append("order-4")
print("처리:", orders.popleft())
print(orders)

scores = [(100, "alpha"), (200, "beta"), (400, "gamma")]
bisect.insort(scores, (300, "delta"))
print(scores)

prices = [105, 98, 110, 102]
heapify(prices)
heappush(prices, 95)
print([heappop(prices) for _ in range(3)])

## 7. 정확한 금융 계산: `Decimal`

부동소수점 오차를 줄이기 위해 정확한 십진수 계산을 사용할 수 있습니다.

In [ ]:
from decimal import Decimal, ROUND_DOWN, getcontext

amount = Decimal("0.70")
rate = Decimal("1.05")
print(round(amount * rate, 2))
print(Decimal("0.1") * 10 == Decimal("1.0"))

getcontext().prec = 28
price = Decimal("105000000.1234")
quantity = Decimal("0.0012")
fee_rate = Decimal("0.0004")
gross = price * quantity
fee = (gross * fee_rate).quantize(Decimal("0.01"), rounding=ROUND_DOWN)
print(gross)
print(fee)

## 8. 실습 검증

핵심 기능들이 실제로 동작하는지 확인합니다.

In [ ]:
from decimal import Decimal
import reprlib
import struct
from string import Template
from collections import deque
import bisect
from heapq import heapify, heappop, heappush

assert "BTC" in reprlib.repr({f"COIN{i}": i for i in range(5)})
assert Template("$symbol 가격은 $price 원입니다.").substitute(symbol="BTC", price="105000000") == "BTC 가격은 105000000 원입니다."
packed = struct.pack("<IQH", 1, 105_000_000, 12)
assert len(packed) == 16
assert Decimal("0.1") + Decimal("0.2") == Decimal("0.3")
assert bisect.bisect_left([10, 20, 30], 25) == 2
heap = [105, 98, 110, 102]
heapify(heap)
heappush(heap, 95)
assert heappop(heap) == 95
assert deque(["order-1", "order-2"]).popleft() == "order-1"
print("Chapter 13 실습 검증 통과")

## 실습 과제

1. `reprlib`로 큰 가격 데이터 구조를 축약해 출력하세요.
2. `Template`으로 사용자 정의 보고서를 만들어 보세요.
3. `struct`로 고정 길이 가격 레코드를 인코딩해 보세요.
4. `threading`과 `Queue`로 여러 코인 가격 수집 작업을 분리해 보세요.
5. `Decimal`으로 거래 수수료를 정확하게 계산해 보세요.